# Тестирование алгоритмов на реальных датасетах

**Datasets:** Iris (150×4), Wine (178×13), Ecoli (327×7), Seeds (210×7), Segmentation (2310×19)

| Тип | Метрика | 
|-----|---------|
| Внешняя | ARI — Adjusted Rand Index | 
| Внешняя | AMI — Adjusted Mutual Information | 
| Внешняя | NMI — Normalized Mutual Information | 
| Внешняя | FMI — Fowlkes-Mallows Index | 
| Внутренняя | SC — Silhouette Coefficient | 
| Внутренняя | CHI — Calinski-Harabasz Index | 
| Внутренняя | DBI — Davies-Bouldin Index |


In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score,
    normalized_mutual_info_score, fowlkes_mallows_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

from data_generator.uci_real import (
    load_iris_fc, load_wine_fc, load_ecoli_fc,
    load_seeds_fc, load_segment_fc,
)
from algorithms import (
    DBSCANWrapper, HDBSCANWrapper,
    DPCWrapper, RDDACWrapper, CKDPCWrapper,
)


In [2]:
DATASETS = [
    load_iris_fc(),
    load_wine_fc(),
    load_ecoli_fc(),
    load_seeds_fc(),
    load_segment_fc(),
]

rows = []
for ds in DATASETS:
    k_true = len(set(ds.y_true.tolist()))
    rows.append({'Датасет': ds.name, 'Объектов (n)': ds.X.shape[0],
                 'Признаков (m)': ds.X.shape[1], 'k (true)': k_true})
display(pd.DataFrame(rows).set_index('Датасет'))

,Объектов (n),Признаков (m),k (true)
Датасет,,,
uci_iris,150,4,3
uci_wine,178,13,3
uci_ecoli,327,7,5
uci_seeds,210,7,3
uci_statlog_segment,2310,19,7


In [3]:
ALGORITHMS = [
    ('DBSCAN',  DBSCANWrapper),
    ('HDBSCAN', HDBSCANWrapper),
    ('DPC',     DPCWrapper),
    ('RD-DAC',  RDDACWrapper),
    ('CKDPC',   CKDPCWrapper),
]
ALG_CLASSES = {name: cls for name, cls in ALGORITHMS}
ALG_NAMES   = [name for name, _ in ALGORITHMS]


In [ ]:
def _arange(a, b, step, dec=3):
    return [round(float(v), dec) for v in np.arange(a, b + step * 0.5, step)]


def compute_all_metrics(X, labels, y_true):
    labels = np.asarray(labels, dtype=int)
    y_true = np.asarray(y_true, dtype=int)
    mask_nn = labels != -1
    k_found = len(set(labels[mask_nn].tolist())) if mask_nn.any() else 0
    noise_pct = float((~mask_nn).sum()) / len(labels) * 100
    n_unique = len(np.unique(labels))
    if k_found >= 2:
        ari = float(adjusted_rand_score(y_true, labels))
        ami = float(adjusted_mutual_info_score(y_true, labels))
        nmi = float(normalized_mutual_info_score(y_true, labels))
        fmi = float(fowlkes_mallows_score(y_true, labels))
    elif k_found == 1:
        ari = ami = nmi = fmi = 0.0
    else:
        ari = ami = nmi = fmi = float('nan')
    X_sub = X[mask_nn]; l_sub = labels[mask_nn]
    if mask_nn.sum() >= 2 and len(np.unique(l_sub)) >= 2:
        try:    sc  = float(silhouette_score(X_sub, l_sub))
        except: sc  = float('nan')
        try:    chi = float(calinski_harabasz_score(X_sub, l_sub))
        except: chi = float('nan')
        try:    dbi = float(davies_bouldin_score(X_sub, l_sub))
        except: dbi = float('nan')
    else:
        sc = chi = dbi = float('nan')
    return dict(k_found=k_found, noise_pct=noise_pct,
                ARI=ari, AMI=ami, NMI=nmi, FMI=fmi, SC=sc, CHI=chi, DBI=dbi)


def _fmt(v, key=''):
    if isinstance(v, float) and (v != v): return '-'
    if key in ('ARI','AMI','NMI','FMI','SC','DBI'): return '{:.4f}'.format(v)
    if key == 'CHI': return '{:.1f}'.format(v)
    return str(v)


def show_ds_table(rows, ds_name, k_true):
    cols = ['k','noise%','ARI','AMI','NMI','FMI','SC','CHI','DBI','best_params']
    print('\n' + ds_name + '  (k_true=' + str(k_true) + ')')
    print('{:<10}'.format('') + ''.join('{:>8}'.format(c) for c in cols[:-1]) + '  params')
    for alg, mets in rows.items():
        p = mets.get('best_params') or {}
        p_str = ', '.join('{0}={1}'.format(k, v) for k, v in p.items())
        row = [str(mets.get('k_found','?')),
               '{:.1f}'.format(mets.get('noise_pct',0.0)),
               _fmt(mets.get('ARI',float('nan')),'ARI'),
               _fmt(mets.get('AMI',float('nan')),'AMI'),
               _fmt(mets.get('NMI',float('nan')),'NMI'),
               _fmt(mets.get('FMI',float('nan')),'FMI'),
               _fmt(mets.get('SC', float('nan')),'SC'),
               _fmt(mets.get('CHI',float('nan')),'CHI'),
               _fmt(mets.get('DBI',float('nan')),'DBI')]
        print('{:<10}'.format(alg) + ''.join('{:>8}'.format(v) for v in row) + '  ' + p_str)


def grid_search(alg_class, param_grid, X, y_true):
    from itertools import product as iproduct
    keys   = list(param_grid.keys())
    combos = list(iproduct(*[param_grid[k] for k in keys])) if keys else [()]
    best_ari = -2.0; best_mets = best_params = None
    for combo in combos:
        params = dict(zip(keys, combo))
        try:
            labels = np.asarray(alg_class(**params).fit_predict(X), dtype=int)
            mets   = compute_all_metrics(X, labels, y_true)
            ari    = mets['ARI'] if (mets['ARI'] == mets['ARI']) else -1.0
            if ari > best_ari:
                best_ari = ari; best_mets = mets; best_params = params
        except Exception:
            pass
    if best_mets is None:
        best_mets = dict(k_found=0, noise_pct=100.0,
                         ARI=float('nan'), AMI=float('nan'), NMI=float('nan'),
                         FMI=float('nan'), SC=float('nan'), CHI=float('nan'), DBI=float('nan'))
    return best_mets, best_params


def get_param_grid(alg_name, X, k_true):
    n, m = X.shape
    if alg_name == 'DBSCAN':
        return {'eps':         _arange(0.01, 1.00, 0.01),
                'min_samples': list(range(1, 51))}
    if alg_name == 'HDBSCAN':
        return {'min_cluster_size': list(range(2, 51, 2)),
                'min_samples':      list(range(1, 11))}
    if alg_name == 'DPC':
        return {'percent': _arange(0.5, 20.0, 0.5)}
    if alg_name == 'RD-DAC':
        return {'k': list(range(2, 31))}
    if alg_name == 'CKDPC':
        if n > 1500:
            return {'alpha':   _arange(0.01, 10.0, 0.10),
                    'percent': _arange(0.5,  10.0, 0.5)}
        return {'alpha':   _arange(0.01, 10.0, 0.05),
                'percent': _arange(0.5,  20.0, 0.5)}
    return {}


In [39]:
ALL_RESULTS = {}

for ds in DATASETS:
    X, y = ds.X, ds.y_true
    k_true = len(set(y.tolist()))
    n, m = X.shape
    print('\n' + ds.name + '  [' + str(n) + ' x ' + str(m) + ']  k_true=' + str(k_true))

    rows = {}
    for alg_name in ALG_NAMES:
        t0 = time.time()
        try:
            pgrid = get_param_grid(alg_name, X, k_true)
            mets, best_p = grid_search(ALG_CLASSES[alg_name], pgrid, X, y)
            elapsed = time.time() - t0
            rows[alg_name] = {**mets, 'best_params': best_p or {}}
            p_str = ', '.join('{0}={1}'.format(k, v) for k, v in (best_p or {}).items() if k != 'n_clusters')
            print('  {:<8}  k={:2d}  noise={:5.1f}%  ARI={:<7}  ({:.1f}s)  [{}]'.format(
                alg_name,
                mets.get('k_found', 0),
                mets.get('noise_pct', 0.0),
                str(round(mets['ARI'], 4)) if not (isinstance(mets['ARI'], float) and mets['ARI'] != mets['ARI']) else '-',
                elapsed,
                p_str))
        except Exception as e:
            print('  {:<8}: ОШИБКА - {}'.format(alg_name, e))
            rows[alg_name] = dict(k_found='ERR', noise_pct=0.0,
                                  ARI=float('nan'), AMI=float('nan'),
                                  NMI=float('nan'), FMI=float('nan'),
                                  SC=float('nan'), CHI=float('nan'), DBI=float('nan'),
                                  best_params={})
    ALL_RESULTS[ds.name] = rows


uci_iris  [150 x 4]  k_true=3
  DBSCAN    k= 2  noise= 46.0%  ARI=0.6246   (77.7s)  [eps=0.13, min_samples=9]
  HDBSCAN   k= 2  noise=  0.0%  ARI=0.5681   (1.0s)  [min_cluster_size=2, min_samples=1]
  DPC       k= 3  noise=  0.0%  ARI=0.8857   (0.1s)  [percent=8.0]
  RD-DAC    k= 4  noise=  0.0%  ARI=0.811    (0.1s)  [k=17]
  CKDPC     k= 6  noise=  6.7%  ARI=0.7486   (42.5s)  [alpha=0.11, percent=0.5]

uci_wine  [178 x 13]  k_true=3
  DBSCAN    k= 2  noise= 57.3%  ARI=0.5379   (74.5s)  [eps=0.51, min_samples=23]
  HDBSCAN   k= 2  noise= 15.2%  ARI=0.3917   (1.0s)  [min_cluster_size=4, min_samples=1]
  DPC       k= 3  noise=  0.0%  ARI=0.6724   (0.1s)  [percent=2.0]
  RD-DAC    k= 3  noise=  0.0%  ARI=0.7269   (0.1s)  [k=22]
  CKDPC     k= 2  noise=  9.0%  ARI=0.4241   (57.6s)  [alpha=0.36, percent=0.5]

uci_ecoli  [327 x 7]  k_true=5
  DBSCAN    k= 3  noise= 18.7%  ARI=0.6632   (92.9s)  [eps=0.23, min_samples=30]
  HDBSCAN   k= 2  noise= 25.1%  ARI=0.4162   (1.8s)  [min_cluster_size=

In [40]:
for ds in DATASETS:
    k_true = len(set(ds.y_true.tolist()))
    show_ds_table(ALL_RESULTS[ds.name], ds.name, k_true)


uci_iris  (k_true=3)
                 k  noise%     ARI     AMI     NMI     FMI      SC     CHI     DBI  params
DBSCAN           2    46.0  0.6246  0.6590  0.6633  0.7533  0.7874   697.7  0.3101  eps=0.13, min_samples=9
HDBSCAN          2     0.0  0.5681  0.7316  0.7337  0.7715  0.6867   502.8  0.3828  min_cluster_size=2, min_samples=1
DPC              3     0.0  0.8857  0.8625  0.8642  0.9233  0.5112   507.4  0.7315  percent=8.0
RD-DAC           4     0.0  0.8110  0.8053  0.8086  0.8712  0.4112   410.1  0.9721  k=17
CKDPC            6     6.7  0.7486  0.7149  0.7249  0.8273 -0.0999   212.5  1.1003  alpha=0.11, percent=0.5

uci_wine  (k_true=3)
                 k  noise%     ARI     AMI     NMI     FMI      SC     CHI     DBI  params
DBSCAN           2    57.3  0.5379  0.5936  0.5982  0.7181  0.4762    95.9  0.5593  eps=0.51, min_samples=23
HDBSCAN          2    15.2  0.3917  0.4718  0.4778  0.6255  0.0466    14.7  2.0314  min_cluster_size=4, min_samples=1
DPC              3     0.0  

In [43]:
ALG_NAMES = [a for a, _ in ALGORITHMS]
KEY_EXT   = ['ARI', 'AMI', 'FMI', 'NMI']
KEY_INT   = ['SC', 'CHI', 'DBI']

col_tuples, data = [], {a: [] for a in ALG_NAMES}
for ds in DATASETS:
    short = ds.name.replace('uci_','').replace('statlog_','').capitalize()
    for metric in KEY_EXT:
        col_tuples.append((short, metric))
        for a in ALG_NAMES:
            v = ALL_RESULTS[ds.name][a].get(metric, np.nan)
            data[a].append(float(v) if isinstance(v, float) else np.nan)

mid = pd.MultiIndex.from_tuples(col_tuples)
df_ext = pd.DataFrame(data, index=mid).T
df_ext.index.name = 'Algorithm'
display(df_ext.style
        .format('{:.4f}', na_rep='-')
        .background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1)
        .set_caption('Внешние метрики'))


In [42]:
col_tuples2, data2 = [], {a: [] for a in ALG_NAMES}
for ds in DATASETS:
    short = ds.name.replace('uci_','').replace('statlog_','').capitalize()
    for metric in KEY_INT:
        col_tuples2.append((short, metric))
        for a in ALG_NAMES:
            v = ALL_RESULTS[ds.name][a].get(metric, np.nan)
            data2[a].append(float(v) if isinstance(v, float) else np.nan)

mid2 = pd.MultiIndex.from_tuples(col_tuples2)
df_int = pd.DataFrame(data2, index=mid2).T
df_int.index.name = 'Algorithm'
display(df_int.style
        .format('{:.4f}', na_rep='-')
        .set_caption('Внутренние метрики'))
